# Migrating to Amazon Bedrock Mantle from OpenAI or Anthropic

> **Sample code — not for production.** Provided as AWS Content under the AWS
> Customer Agreement; do not use it in production accounts or on production or other
> critical data. Running these cells calls Amazon Bedrock and incurs charges. Full
> disclaimer in the [README](../README.md#disclaimer).

The headline promise is real: point your existing SDK at a new base URL and key,
and it works. This notebook shows that — and then shows the four places where a
naive port breaks, so you find them here rather than in production.

## What this notebook covers
- The two-line change (OpenAI SDK and Anthropic SDK)
- The four real breakages: path prefixes, sampling params, tiers, absent APIs
- A compatibility shim that makes one client work across families
- LLM-gateway configuration
- A migration checklist you can work through

## Self-contained, but see also
- **Auth, the three URL paths** →
  `../00-foundations/01-endpoints-auth-and-the-three-paths.ipynb`
- **Live capability survey** → `01-choosing-a-model-and-api.ipynb`

## Prerequisites
```bash
pip install -r ../requirements.txt
```

Needs openai, anthropic, aws-bedrock-token-generator.

`requirements.txt` pins the exact versions this collection was tested
against. An unpinned install resolves whatever is current, which may be
untested or compromised (OWASP LLM03, Supply Chain).

### Where the helpers come from

The next cell does this:

```python
sys.path.insert(0, "../_shared")
from bedrock import ...
```

`bedrock` is **not** a package from PyPI — it is this collection's own helper
module, [`_shared/bedrock.py`](../_shared/bedrock.py). Every notebook sits one
level down, so `../_shared` puts it on the import path. It exists only to remove
repetition; the notebooks are the teaching material, and nothing in the module is
required to call Bedrock yourself.

What this notebook uses from it:

| Helper | What it does |
|---|---|
| `err` | pulls the human-readable message out of an error body, redacted |
| `post` | signed JSON HTTP with retries. **Never raises on 4xx/5xx** — it returns `(status, body)` so a cell can *show* an error instead of stopping the notebook |
| `response_text` | assistant text from a Responses API payload |
| `safe_print` | `print()` with account IDs, IAM principals and opaque service IDs redacted |
| `stream_lines` | raw SSE lines from a streaming endpoint, no SDK |

Two behaviours worth knowing before you read any output below, because several
cells depend on them:

- **`post()` and `converse()` never raise on a service error.** They return the
  status and body so a cell can *show* a 400 rather than stopping the notebook.
  Many cells here deliberately provoke an error to demonstrate a limit.
- **Anything printed from a control-plane response goes through redaction**, since
  this output is committed to a public repository.


In [1]:
import json
import sys

sys.path.insert(0, "../_shared")
from bedrock import err, post, response_text, safe_print

REGION = "us-east-1"

# One survey list, shared by every probe below. Six models across three wire
# protocols and all three mantle path prefixes.
SURVEY = [
    "openai.gpt-5.6-sol",
    "openai.gpt-oss-120b",
    "google.gemma-4-31b",
    "xai.grok-4.3",
    "qwen.qwen3-32b",
    "anthropic.claude-haiku-4-5",
]
print("region:", REGION, "|", len(SURVEY), "models surveyed")

region: us-east-1 | 6 models surveyed


## 1. The two-line change

For an OpenAI codebase, `base_url` and `api_key` are the entire migration.

In [2]:
from aws_bedrock_token_generator import provide_token
from openai import OpenAI

# BEFORE (OpenAI):
#   client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
#
# AFTER (Bedrock Mantle):
client = OpenAI(
    api_key=provide_token(region=REGION),  # short-term Bedrock key from IAM
    base_url=f"https://bedrock-mantle.{REGION}.api.aws/openai/v1",
)

response = client.responses.create(
    model="openai.gpt-5.6-sol",  # note: Bedrock model IDs, not OpenAI's
    input="Confirm you are reachable in five words.",
    max_output_tokens=60,
)
print(response.output_text)

Yes, I am reachable now.


Environment-variable form, if you would rather not touch code at all:

```bash
export OPENAI_BASE_URL="https://bedrock-mantle.us-east-1.api.aws/openai/v1"
export OPENAI_API_KEY="$(
  python -c 'from aws_bedrock_token_generator import provide_token
print(provide_token(region="us-east-1"))'
)"
```

**Do not** hardcode a long-term key. Short-term keys are presigned SigV4 (AWS
 Signature Version 4), expire
within 12 hours, and inherit the permissions of the role that minted them.

In [3]:
# The Anthropic SDK migrates the same way. Note the base URL omits /v1 —
# the SDK appends it.
import anthropic

claude = anthropic.Anthropic(
    api_key=provide_token(region=REGION),
    base_url=f"https://bedrock-mantle.{REGION}.api.aws/anthropic",
)
message = claude.messages.create(
    model="anthropic.claude-haiku-4-5",
    max_tokens=60,  # required on the Messages API
    messages=[{"role": "user", "content": "Confirm you are reachable in five words."}],
)
print("".join(b.text for b in message.content if b.type == "text"))

I am here and ready now.


## 2. Model IDs are not OpenAI's

`gpt-4o` does not exist here. Map your model names first — this is the most common
first-day error.

In [4]:
MODEL_MAP = {
    # OpenAI name           -> a reasonable Bedrock Mantle equivalent
    "gpt-4o": "openai.gpt-5.6-sol",
    "gpt-4o-mini": "openai.gpt-5.4",
    "gpt-4-turbo": "openai.gpt-5.5",
    "o1": "openai.gpt-5.6-sol",
    "gpt-3.5-turbo": "openai.gpt-oss-20b",
    # Anthropic
    "claude-3-5-sonnet": "anthropic.claude-sonnet-5",
    "claude-3-haiku": "anthropic.claude-haiku-4-5",
}
for old, new in MODEL_MAP.items():
    print(f"  {old:22} -> {new}")

  gpt-4o                 -> openai.gpt-5.6-sol
  gpt-4o-mini            -> openai.gpt-5.4
  gpt-4-turbo            -> openai.gpt-5.5
  o1                     -> openai.gpt-5.6-sol
  gpt-3.5-turbo          -> openai.gpt-oss-20b
  claude-3-5-sonnet      -> anthropic.claude-sonnet-5
  claude-3-haiku         -> anthropic.claude-haiku-4-5


In [5]:
# Prove the point: an OpenAI model ID is simply not present.
code, data = post(
    "/openai/v1/responses",
    {"model": "gpt-4o", "input": "Hi", "max_output_tokens": 16},
    region=REGION,
    attempts=1,
    timeout=45,
)
print(f"model='gpt-4o' -> HTTP {code}: {err(data)[:100]}")

model='gpt-4o' -> HTTP 404: The model 'gpt-4o' does not exist


## 3. Breakage 1 — routing: prefix, API surface, and budget field

There is no single base URL that serves every model. Three prefixes exist, and the
OpenAI family is split across two of them. Two more routing decisions ride along
with the prefix and are just as easy to get wrong:

- **Which API the model actually serves.** Most open-weight families have no
  Responses API, and the gpt-oss *safeguard* variants lack it even though base
  gpt-oss has it.
- **What the token budget is called.** `max_output_tokens` on Responses,
  `max_tokens` on Messages and most of Chat Completions — but
  `max_completion_tokens` for gpt-5.6 on Chat Completions.

The helpers below answer all three, and every probe in this notebook uses them. §7's
compatibility shim composes them too, so there is one place to correct.

In [6]:
# The four routing questions, answered per model. Everything below in this notebook
# uses these, and section 7's compatibility shim composes them rather than
# restating them.

# Families with no Responses API: Chat Completions only. Note gpt-oss-safeguard is
# here while base gpt-oss is not -- the provider prefix is not enough to decide.
CHAT_ONLY = (
    "qwen.", "deepseek.", "zai.", "minimax.", "moonshotai.", "mistral.",
    "nvidia.", "writer.", "openai.gpt-oss-safeguard", "google.gemma-3",
)
# Chat Completions wants max_completion_tokens rather than max_tokens for these.
COMPLETION_TOKENS = ("openai.gpt-5.6",)
# Reasoning-first models spend the budget thinking before any answer text, so a
# small cap returns HTTP 200 and an empty string.
REASONS_FIRST = ("xai.", "openai.gpt-5.6", "moonshotai.kimi-k2-thinking")


def api_prefix(model_id: str) -> str:
    """Which of the three URL prefixes serves this model."""
    if model_id.startswith("anthropic."):
        return "/anthropic/v1"
    if model_id.startswith(("google.gemma-4", "openai.gpt-5", "xai.")):
        return "/openai/v1"
    return "/v1"


def surface(model_id: str) -> str:
    """Which API this model actually serves: messages / chat / responses."""
    if model_id.startswith("anthropic."):
        return "messages"
    if model_id.startswith(CHAT_ONLY):
        return "chat"
    return "responses"


def budget_field(model_id: str) -> str:
    """The token-budget parameter this model+API expects.

    Getting this wrong returns a 400 that reads exactly like "this API does not
    exist here", which is the trap section 6 is about.
    """
    which = surface(model_id)
    if which == "responses":
        return "max_output_tokens"          # minimum 16
    if which == "chat" and model_id.startswith(COMPLETION_TOKENS):
        return "max_completion_tokens"
    return "max_tokens"                     # Messages: required, no default


def build_request(model_id, prompt="Hi", *, budget=16, system=None,
                  sampling=None, tier=None):
    """Return (path, body, headers) in whichever wire format this model wants."""
    which, prefix = surface(model_id), api_prefix(model_id)
    floor = 1000 if model_id.startswith(REASONS_FIRST) else 16
    body = {"model": model_id, budget_field(model_id): max(floor, budget)}
    body.update(sampling or {})
    headers = {}

    if which == "messages":
        # service_tier is not a Messages parameter at all, so it is never sent.
        headers["anthropic-version"] = "2023-06-01"
        body["messages"] = [{"role": "user", "content": prompt}]
        if system:
            body["system"] = system
        return f"{prefix}/messages", body, headers

    if tier:
        body["service_tier"] = tier
    turns = ([{"role": "system", "content": system}] if system else []) + [
        {"role": "user", "content": prompt}
    ]
    if which == "chat":
        body["messages"] = turns
        return f"{prefix}/chat/completions", body, headers
    body["input"] = turns
    body["store"] = False
    return f"{prefix}/responses", body, headers


print(f"{'model':30} {'prefix':16} {'surface':10} budget field")
print("-" * 74)
for mid in SURVEY:
    print(f"{mid:30} {api_prefix(mid):16} {surface(mid):10} {budget_field(mid)}")

model                          prefix           surface    budget field
--------------------------------------------------------------------------
openai.gpt-5.6-sol             /openai/v1       responses  max_output_tokens
openai.gpt-oss-120b            /v1              responses  max_output_tokens
google.gemma-4-31b             /openai/v1       responses  max_output_tokens
xai.grok-4.3                   /openai/v1       responses  max_output_tokens
qwen.qwen3-32b                 /v1              chat       max_tokens
anthropic.claude-haiku-4-5     /anthropic/v1    messages   max_tokens


In [7]:
# One client with the wrong prefix fails for half your models.
wrong = OpenAI(
    api_key=provide_token(region=REGION),
    base_url=f"https://bedrock-mantle.{REGION}.api.aws/openai/v1",
)
for mid in ("openai.gpt-5.6-sol", "openai.gpt-oss-120b"):
    try:
        wrong.responses.create(model=mid, input="Hi", max_output_tokens=16)
        print(f"  {mid:24} -> ok on /openai/v1")
    except Exception as exc:
        print(f"  {mid:24} -> {type(exc).__name__}: {str(exc)[:80]}")

  openai.gpt-5.6-sol       -> ok on /openai/v1


  openai.gpt-oss-120b      -> BadRequestError: Error code: 400 - {'error': {'code': 'validation_error', 'message': "The model '


## 4. Breakage 2 — sampling parameters are per model

A shared `temperature=0.7, top_p=0.95` default, which is completely normal in
OpenAI code, fails on several models here.

In [8]:
# Probe each model on the API it ACTUALLY serves, using the routing from section 3.
# The earlier version of this cell always POSTed /responses, so every row for a
# Chat-Completions-only model was a 400 about the missing API -- and the paragraph
# below it then contradicted its own table. Blaming the parameter for a missing API
# is the same mistake in reverse.
CASES = [
    ("0.7+top_p", {"temperature": 0.7, "top_p": 0.95}),
    ("temp 0.7", {"temperature": 0.7}),
    ("temp 1.0", {"temperature": 1.0}),
    ("top_p", {"top_p": 0.95}),
]

print(f"{'model':26} {'surface':10}" + "".join(f"{h:>11}" for h, _ in CASES))
print("-" * 78)
param_support = {}
for mid in SURVEY:
    row = []
    for _, extra in CASES:
        path, body, headers = build_request(mid, sampling=extra)
        code, _ = post(path, body, region=REGION, attempts=1, timeout=60,
                       headers=headers or None)
        row.append("ok" if code == 200 else str(code))
    param_support[mid] = row
    print(f"{mid:26} {surface(mid):10}" + "".join(f"{v:>11}" for v in row))

model                      surface     0.7+top_p   temp 0.7   temp 1.0      top_p
------------------------------------------------------------------------------


openai.gpt-5.6-sol         responses         400        400         ok        400


openai.gpt-oss-120b        responses          ok         ok         ok         ok


google.gemma-4-31b         responses          ok         ok         ok         ok


xai.grok-4.3               responses          ok         ok         ok         ok


qwen.qwen3-32b             chat               ok         ok         ok         ok


anthropic.claude-haiku-4-5 messages          400         ok         ok         ok


Read the table, not a remembered rule — which models refuse what has changed
twice during this collection's life. What is durable:

- **There is no single sampling config that works everywhere.** At the time of
  writing the GPT-5.5 and GPT-5.6 families accept `temperature` only at its default
  `1.0` and refuse `top_p` outright; newer Claude models reject both as
  *deprecated*; `claude-haiku-4-5` accepts either one but **not both together**.
  Gemma 4 and Grok have been in and out of that set.
- **The safest port is to send neither parameter**, then add each back per model
  after testing.
- **The service names the parameter it refused.** That is what makes the shim in §7
  able to recover automatically instead of carrying a table that ages.

## 5. Breakage 3 — `service_tier` is not universal

In [9]:
print(f"{'model':26} {'surface':10} {'flex':>8} {'priority':>10}")
print("-" * 58)
for mid in SURVEY:
    if surface(mid) == "messages":
        # service_tier is not a Messages parameter, so there is nothing to test.
        # Printing "400" here would imply the tier was refused on its merits.
        print(f"{mid:26} {'messages':10} {'n/a':>8} {'n/a':>10}")
        continue
    row = []
    for tier in ("flex", "priority"):
        path, body, headers = build_request(mid, tier=tier)
        code, _ = post(path, body, region=REGION, attempts=1, timeout=60,
                       headers=headers or None)
        row.append("ok" if code == 200 else str(code))
    print(f"{mid:26} {surface(mid):10} {row[0]:>8} {row[1]:>10}")

model                      surface        flex   priority
----------------------------------------------------------


openai.gpt-5.6-sol         responses       400        400


openai.gpt-oss-120b        responses        ok         ok


google.gemma-4-31b         responses        ok         ok


xai.grok-4.3               responses        ok         ok


qwen.qwen3-32b             chat             ok         ok
anthropic.claude-haiku-4-5 messages        n/a        n/a


## 6. Breakage 4 — the API you use may not exist for your model

If your codebase is built on Responses, note that most open-weight families do not
serve it. The reverse trap is subtler: **gpt-5.6 *does* serve Chat Completions**,
but it refuses `max_tokens` in favour of `max_completion_tokens`, so a naive probe
gets a 400 and concludes the API is missing. An earlier version of this notebook
made exactly that mistake and recorded it as fact in three places.

The rule: send both candidate budget fields before deciding an API is absent, and
read *which* thing the error names — the path, or the parameter.

In [10]:
# Try BOTH budget parameter names before concluding an API is missing. gpt-5.6
# serves Chat Completions but refuses `max_tokens`, and reading that 400 as "no
# Chat Completions" is how a false claim reached three places in this notebook.
print(f"{'model':26} {'Responses':>10} {'ChatCompl':>10} {'CC budget field':>22}")
print("-" * 72)
for mid in SURVEY:
    if mid.startswith("anthropic."):
        print(f"{mid:26} {'n/a':>10} {'n/a':>10} {'Messages only':>22}")
        continue
    prefix = api_prefix(mid)
    code_r, _ = post(
        f"{prefix}/responses",
        {"model": mid, "input": "Hi", "max_output_tokens": 16},
        region=REGION, attempts=1, timeout=60,
    )
    field, code_c = None, None
    for candidate in ("max_tokens", "max_completion_tokens"):
        code_c, _ = post(
            f"{prefix}/chat/completions",
            {"model": mid, "messages": [{"role": "user", "content": "Hi"}],
             candidate: 16},
            region=REGION, attempts=1, timeout=60,
        )
        if code_c == 200:
            field = candidate
            break
    print(f"{mid:26} {code_r:>10} {code_c:>10} {(field or '-'):>22}")

model                       Responses  ChatCompl        CC budget field
------------------------------------------------------------------------


openai.gpt-5.6-sol                200        200  max_completion_tokens


openai.gpt-oss-120b               200        200             max_tokens


google.gemma-4-31b                200        200             max_tokens


xai.grok-4.3                      200        200             max_tokens


qwen.qwen3-32b                    400        200             max_tokens
anthropic.claude-haiku-4-5        n/a        n/a          Messages only


## 7. A compatibility shim

One class that resolves path, API, parameters and tier per model, so your
application code stops caring.

In [11]:
class MantleCompat:
    """Model-agnostic wrapper over the three bedrock-mantle API surfaces.

    Routing comes from section 3 -- `surface()`, `budget_field()`,
    `build_request()`. What this class adds is the **error-driven retry**, and that
    is the part worth copying.

    An earlier version carried a static table of which models reject which
    parameters. That table was correct when written and wrong within weeks -- Gemma 4
    and Grok both changed twice -- and the class failed on half the models it
    claimed to handle. Since the service *names* the offending parameter in its 400
    (section 4), the robust move is to drop that parameter and retry rather than to
    remember a rule.
    """

    TUNABLE = ("temperature", "top_p", "service_tier")

    def __init__(self, region=REGION, project=None):
        self.region, self.project = region, project

    def prefix(self, model):
        return api_prefix(model)

    def surface(self, model):
        return surface(model)

    def _project_header(self, which):
        if not self.project:
            return {}
        key = "anthropic-workspace" if which == "messages" else "OpenAI-Project"
        return {key: self.project}

    @staticmethod
    def _offending_key(message):
        """The parameter the service just refused, if it named one.

        Mantle is consistent about naming it. Three shapes seen in practice:
          Unsupported parameter: 'top_p' is not supported with this model.
          `temperature` is deprecated for this model.
          unsupported service_tier 'flex'                  <- field unquoted
        """
        import re

        text = message or ""
        quoted = re.findall(r"[\'`\"]([a-z_]+)[\'`\"]", text)
        for name in quoted:
            if name in MantleCompat.TUNABLE:
                return name
        # A bare mention is only safe to act on when the message is not about a
        # path: "does not support the '/v1/responses' API" names no parameter.
        if "API" not in text:
            for name in MantleCompat.TUNABLE:
                if name in text:
                    return name
        return None

    def complete(self, model, prompt, *, system=None, max_tokens=400,
                 temperature=None, top_p=None, tier="default", timeout=120,
                 attempts=2):
        """One call surface over all three APIs, self-healing on refused parameters.

        `timeout` and `attempts` are bounded on purpose. Reasoning-first models can
        take many minutes for one call under load, and post()'s defaults (240s x 5)
        would turn one slow model into a 20-minute stall for the whole loop.
        """
        sampling = {}
        if temperature is not None:
            sampling["temperature"] = temperature
        if top_p is not None:
            sampling["top_p"] = top_p
        current_tier = tier
        dropped = []

        for _ in range(len(self.TUNABLE) + 1):
            path, body, headers = build_request(
                model, prompt, budget=max_tokens, system=system,
                sampling=sampling, tier=current_tier,
            )
            headers.update(self._project_header(surface(model)))
            code, data = post(path, body, region=self.region,
                              headers=headers or None, timeout=timeout,
                              attempts=attempts)
            if code == 200:
                if dropped:
                    print(f"      [{model}: dropped {', '.join(dropped)} and retried]")
                return self._extract(surface(model), data)

            refused = self._offending_key(err(data))
            if refused == "service_tier" and current_tier != "default":
                current_tier, _ = "default", dropped.append("service_tier")
                continue
            if refused in sampling:
                sampling.pop(refused)
                dropped.append(refused)
                continue
            raise RuntimeError(f"{model}: HTTP {code}: {err(data)}")
        raise RuntimeError(f"{model}: still refused after dropping {dropped}")

    @staticmethod
    def _extract(which, data):
        if which == "messages":
            return "".join(
                b.get("text", "")
                for b in data.get("content", [])
                if b.get("type") == "text"
            )
        if which == "chat":
            return (data.get("choices") or [{}])[0].get("message", {}).get(
                "content"
            ) or ""
        return response_text(data)


compat = MantleCompat()
PROMPT = "Name one benefit of a managed inference endpoint. One sentence."

Exercise the shim across every path family — one call site, six models, three
different wire protocols underneath. Watch for `[dropped ...]` notes: that is the
shim reading a 400, removing the parameter the service named, and retrying.

In [12]:
print(f"{'model':30} {'surface':10}  answer")
print("-" * 96)
for mid in SURVEY:
    try:
        # Deliberately pass BOTH sampling params and a flex tier. The shim sends
        # them, reads any 400 that names one, drops it and retries -- so watch for
        # the "[dropped ...]" notes.
        answer = compat.complete(
            mid, PROMPT, temperature=0.7, top_p=0.95, tier="flex", max_tokens=200
        )
        shown = " ".join(answer.split())[:52] or "(empty - raise max_tokens)"
        print(f"{mid:30} {compat.surface(mid):10}  {shown!r}")
    except RuntimeError as exc:
        print(f"{mid:30} {compat.surface(mid):10}  FAILED: {str(exc)[:52]}")

model                          surface     answer
------------------------------------------------------------------------------------------------


      [openai.gpt-5.6-sol: dropped service_tier, temperature, top_p and retried]
openai.gpt-5.6-sol             responses   'A managed inference endpoint simplifies deployment, '


openai.gpt-oss-120b            responses   'A managed inference endpoint automatically handles s'


google.gemma-4-31b             responses   'A managed inference endpoint removes the operational'


xai.grok-4.3                   responses   'Managed inference endpoints automatically handle inf'


qwen.qwen3-32b                 chat        'A managed inference endpoint provides automatic scal'


      [anthropic.claude-haiku-4-5: dropped temperature and retried]
anthropic.claude-haiku-4-5     messages    'A managed inference endpoint automatically handles s'


Same call signature, six models, three different API surfaces. Where a parameter
was refused the shim dropped it and retried rather than failing, which is why the
answers come back even though the call site passed `temperature`, `top_p` and
`flex` to every model indiscriminately.

That recovery is the part to copy. A shim built on a table of per-model
restrictions works until the table is stale; one that reads the error the service
actually returned keeps working.

## 8. Streaming across surfaces

The three APIs stream differently. Normalise it once.

In [13]:
from bedrock import stream_lines


def stream_text(model, prompt, region=REGION, max_tokens=200):
    """Yield text deltas from whichever streaming shape this model uses."""
    helper = MantleCompat(region=region)
    prefix, surface = helper.prefix(model), helper.surface(model)
    headers = {"anthropic-version": "2023-06-01"} if surface == "messages" else None

    if surface == "messages":
        path = f"{prefix}/messages"
        body = {
            "model": model,
            "max_tokens": max_tokens,
            "stream": True,
            "messages": [{"role": "user", "content": prompt}],
        }
    elif surface == "chat":
        path = f"{prefix}/chat/completions"
        body = {
            "model": model,
            "max_tokens": max_tokens,
            "stream": True,
            "messages": [{"role": "user", "content": prompt}],
        }
    else:
        path = f"{prefix}/responses"
        body = {
            "model": model,
            "max_output_tokens": max(16, max_tokens),
            "stream": True,
            "input": prompt,
        }

    for line in stream_lines(path, body, region=region, headers=headers):
        if not line.startswith("data: "):
            continue
        payload = line[6:].strip()
        if payload == "[DONE]":
            return
        try:
            event = json.loads(payload)
        except json.JSONDecodeError:
            continue
        if surface == "messages":
            if event.get("type") == "content_block_delta":
                delta = event.get("delta", {})
                if delta.get("type") == "text_delta":
                    yield delta.get("text", "")
        elif surface == "chat":
            for choice in event.get("choices", []):
                chunk = (choice.get("delta") or {}).get("content")
                if chunk:
                    yield chunk
        else:
            if event.get("type") == "response.output_text.delta":
                yield event.get("delta", "")


for mid in ("google.gemma-4-31b", "qwen.qwen3-32b", "anthropic.claude-haiku-4-5"):
    print(f"--- {mid} ---")
    chunks = 0
    for piece in stream_text(mid, "Count from one to five.", max_tokens=80):
        chunks += 1
        print(piece, end="", flush=True)
    print(f"\n[{chunks} deltas]\n")

--- google.gemma-4-31b ---


One

,

 two

,

...

One, two, three, four, five.


[1 deltas]

--- anthropic.claude-haiku-4-5 ---


1


2
3
4
5


[2 deltas]



## 9. LLM gateways

Because the endpoint is OpenAI-compatible, gateways such as LiteLLM work by
configuring a custom base URL. The two things to get right are the **per-model
base URL** (path prefix) and **token refresh**.

```yaml
model_list:
  - model_name: gpt-frontier
    litellm_params:
      model: openai/openai.gpt-5.6-sol
      api_base: https://bedrock-mantle.us-east-1.api.aws/openai/v1
      api_key: os.environ/BEDROCK_API_KEY

  - model_name: gpt-oss
    litellm_params:
      model: openai/openai.gpt-oss-120b
      api_base: https://bedrock-mantle.us-east-1.api.aws/v1   # bare /v1
      api_key: os.environ/BEDROCK_API_KEY

  - model_name: qwen
    litellm_params:
      model: openai/qwen.qwen3-32b
      api_base: https://bedrock-mantle.us-east-1.api.aws/v1
      api_key: os.environ/BEDROCK_API_KEY
```

Refresh the key on a schedule — it expires within 12 hours and cannot be renewed:

```bash
# cron / sidecar
export BEDROCK_API_KEY="$(python -c '
from aws_bedrock_token_generator import provide_token
print(provide_token(region="us-east-1"))')"
```

In [14]:
# Verify the exact base URLs a gateway config would need.
print("base URLs by model (paste into your gateway config):")
for mid in (
    "openai.gpt-5.6-sol",
    "openai.gpt-oss-120b",
    "google.gemma-4-31b",
    "qwen.qwen3-32b",
    "anthropic.claude-haiku-4-5",
):
    print(
        f"  {mid:30} https://bedrock-mantle.{REGION}.api.aws"
        f"{MantleCompat().prefix(mid)}"
    )

base URLs by model (paste into your gateway config):
  openai.gpt-5.6-sol             https://bedrock-mantle.us-east-1.api.aws/openai/v1
  openai.gpt-oss-120b            https://bedrock-mantle.us-east-1.api.aws/v1
  google.gemma-4-31b             https://bedrock-mantle.us-east-1.api.aws/openai/v1
  qwen.qwen3-32b                 https://bedrock-mantle.us-east-1.api.aws/v1
  anthropic.claude-haiku-4-5     https://bedrock-mantle.us-east-1.api.aws/anthropic/v1


## 10. Features you gain, and features you lose

| Moving from | You gain | You lose / must change |
|---|---|---|
| OpenAI API | IAM auth, Projects, ZDR (zero data retention), AWS-hosted Web Search, service tiers | OpenAI model IDs; some sampling params; per-model budget-field names |
| Anthropic API | IAM auth, Workspaces, `count_tokens` on mantle | `output_config.format` (use forced tools); `temperature` on newer models |
| `bedrock-runtime` | OpenAI/Anthropic-shaped APIs, stateful chat, server-side tools | cross-Region inference, Provisioned Throughput, batch |

In [15]:
# The gains are real — here is Projects-based attribution, which has no OpenAI
# equivalent.
code, project = post(
    "/v1/organization/projects",
    {
        "name": "migration-samples",
        "tags": {
            "Application": "MigrationDemo",
            "Environment": "Demo",
            "CostCenter": "0000",
        },
    },
    region=REGION,
)
project_id = project.get("id")
safe_print("project:", code, project_id)

attributed = MantleCompat(project=project_id)
print(
    "attributed call:",
    attributed.complete("google.gemma-4-31b", "Reply OK.", max_tokens=20)[:40],
)

code, archived = post(
    f"/v1/organization/projects/{project_id}/archive", {}, region=REGION
)
print("archived:", code, archived.get("status"))

project: 200 proj_iujq2odv...


attributed call: OK


archived: 200 archived


## 11. Migration checklist

1. **Map model IDs.** OpenAI/Anthropic names do not exist here.
2. **Resolve the path prefix per model.** One base URL will not cover your fleet.
3. **Strip shared sampling defaults.** Add `temperature`/`top_p` back per model.
4. **Check the API surface exists** for each model (Responses vs Chat vs Messages).
5. **Gate `service_tier`** — gpt-5.x is `default`-only, and it is not a
   Messages parameter at all.
6. **Set `store` explicitly.** It defaults to `true` = 30-day retention.
7. **Replace API-key handling** with short-term token minting plus refresh.
8. **Add retry/backoff** — mantle has no RPM quota and sheds load.
9. **Set client timeouts** — a wrong path can stall rather than 400.
10. **Parse output defensively** — 200 does not guarantee the constraint held.
11. **Re-point observability** to the `AWS/BedrockMantle` namespace.
12. **Create Projects** and tag them for cost attribution.
13. **Confirm Region coverage** for every model you depend on.

## Gotchas — migration

| Gotcha | Detail |
|---|---|
| Model IDs | `gpt-4o` etc. do not exist; map them first |
| One base URL is not enough | Three prefixes, and the OpenAI family spans two |
| Shared sampling defaults | No universal config. gpt-5.5/5.6 want `temperature=1.0` and refuse `top_p`; newer Claude rejects both; `haiku-4-5` takes either but not both |
| `service_tier` | gpt-5.x accepts `default` only |
| Budget field name | Chat Completions wants `max_completion_tokens` on gpt-5.6, `max_tokens` elsewhere. A 400 here reads like a missing API |
| Responses coverage | Absent on most open-weight families, and on the gpt-oss **safeguard** variants even though base gpt-oss serves it |
| `store` default | `true` — 30-day retention unless you opt out |
| Token lifetime | ≤12 h, not refreshable, Region-pinned |
| `max_output_tokens` | Minimum 16 on Responses |
| Long-term keys | Exploration only; they create a static IAM user credential |

## Next
`03-production-hardening-checklist.ipynb` — everything to verify before launch.